# Documentos de carta enriquecidos com o dicionario de keywords

*(Feito com ClaudeAI: https://claude.ai/share/b3536ad5-de2f-4b3f-a788-ba1c9a17f7da)*

Junta o texto de cada carta (do bulk data "Oracle Cards" da Scryfall) com
as definicoes do `keyword_dictionary.json`, gerando as representacoes de
texto usadas na etapa de embedding, mais os dados estruturados relevantes
pro jogo (nome, custo, cor, poder/resistencia/lealdade quando aplicavel).

**Revisado apos auditoria externa E uma segunda rodada de simplificacao**
(motivos detalhados em cada celula). Mudancas desta rodada:
- Cartas que nao sao cartas de baralho de verdade (tokens, emblemas,
  cartas so de arte do Secret Lair, objetos de Vanguard/Archenemy/
  Planechase) sao filtradas fora antes de processar -- ficam soh cartas
  jogaveis em baralho (Unstable/Un-sets ficam, por decisao explicita).
- `id`/`oracle_id` (identificadores internos da Scryfall) removidos -- o
  nome da carta passa a ser a referencia. **Aviso:** existem ~15 cartas
  do Unstable com o MESMO nome e textos DIFERENTES de proposito (gimmick
  do set, nao e' bug); o arquivo continua sendo uma lista simples (nao um
  dicionario indexado por nome), entao nada se perde, mas cuidado se um
  dia construir um `{nome: carta}` em algum lugar -- so a ultima versao
  de cada nome duplicado sobreviveria a essa estrutura.
- `keyword_definitions` (lista estruturada) removida do registro final --
  ainda calculada internamente (necessaria pra montar o texto de
  embedding), so nao e' mais salva, ja que o conteudo dela ja esta
  dentro do texto estruturado.
- Ficam so 2 das 3 variantes de texto: `embedding_text_structured` (a
  usada nos embeddings) e `embedding_text_baseline` (controle mais barato
  pra comparacao futura). `embedding_text_expanded` foi removida.

Basta ajustar os caminhos na celula de configuracao e rodar tudo
(`Run All`).

## Configuracao

In [1]:
import json
import re

CARDS_PATH = "Dados/Raw/oracle-cards.jsonl"
KEYWORD_DICT_PATH = "Dados/Etapa1_card_expanded/keyword_dictionary.json"
ABILITY_WORDS_PATH = "Dados/Etapa1_card_expanded/keyword_dictionary_ability_words.json"
OUTPUT_PATH = "Dados/Etapa1_card_expanded/card_documents.jsonl"

# layouts que NAO sao cartas de baralho de verdade -- tokens, emblemas,
# cartas so de arte (Secret Lair "art_series"), e objetos de formatos
# alternativos (Vanguard/Archenemy/Planechase) nao fazem sentido num
# corpus de busca pra montar deck.
EXCLUDED_LAYOUTS = {
    "token", "double_faced_token", "emblem", "art_series",
    "vanguard", "scheme", "planar", "front_card",
}

# Un-sets (bordas prateadas -- Unglued/Unhinged/Unstable/Unfinity etc.)
# excluidos por set_type, nao por layout: cartas como "Everythingamajig"
# usam layout "normal" mas tem o MESMO nome com textos diferentes de
# proposito (gimmick de design), o que so o filtro por set_type pega
# por completo.
EXCLUDED_SET_TYPES = {"funny"}

## Extrair o texto da carta (e preservar as faces)

Cartas normais tem `oracle_text` no nivel principal. Cartas de dupla face
(layout `transform`, `modal_dfc`, etc) **nao tem** -- o texto fica dentro de
`card_faces`. A versao anterior so concatenava as faces com `" // "` e
perdia a estrutura; agora `get_card_faces()` devolve as faces como lista
estruturada (nome/type_line/mana_cost/oracle_text de cada uma), e
`get_card_text()` continua gerando a versao concatenada pra quem preferir
um unico texto.

In [2]:
def get_card_faces(card: dict):
    faces = card.get("card_faces")
    if not faces:
        return None
    return [
        {
            "name": f.get("name"),
            "type_line": f.get("type_line"),
            "mana_cost": f.get("mana_cost"),
            "oracle_text": f.get("oracle_text"),
            "power": f.get("power"),
            "toughness": f.get("toughness"),
        }
        for f in faces
    ]


def get_card_text(card: dict) -> str:
    if card.get("oracle_text"):
        return card["oracle_text"]
    faces = card.get("card_faces")
    if faces:
        parts = [f.get("oracle_text", "") for f in faces if f.get("oracle_text")]
        return " // ".join(parts)
    return ""

## Decodificar simbolos de mana e ativacao

`oracle_text` e `mana_cost` usam simbolos tipo `{T}` (virar), `{G}` (mana
verde), `{2/W}` (hibrido monocolorido) -- um modelo de embedding generico
nao tem por que saber o que esses simbolos significam, sao notacao
especifica do jogo. A regra 107 da propria Comprehensive Rules define
todos eles (107.4 pros simbolos de mana, 107.5/107.6 pro simbolo de virar/
desvirar), entao construimos o glossario de simbolos a partir dai, em vez
de eu confiar so na minha memoria sobre o assunto.

**Importante: isso so afeta as variantes `embedding_text_*`.** O campo
`oracle_text` continua com os simbolos originais intactos -- a decodificacao
e' so uma transformacao pra ajudar o embedding, nao uma edicao dos dados
de origem (mesmo principio de manter raw + versao processada que usamos
no dicionario de keywords).

In [3]:
COLOR_NAMES = {"W": "white", "U": "blue", "B": "black", "R": "red", "G": "green"}
HYBRID_PAIRS = [("W","U"),("W","B"),("U","B"),("U","R"),("B","R"),
                ("B","G"),("R","G"),("R","W"),("G","W"),("G","U")]

SYMBOL_GLOSSARY = {}
for c, name in COLOR_NAMES.items():
    SYMBOL_GLOSSARY[c] = f"{name} mana"
SYMBOL_GLOSSARY["C"] = "colorless mana"
SYMBOL_GLOSSARY["S"] = "snow mana"
SYMBOL_GLOSSARY["X"] = "X generic mana"
SYMBOL_GLOSSARY["T"] = "tap this permanent"
SYMBOL_GLOSSARY["Q"] = "untap this permanent"
for n in range(0, 21):
    SYMBOL_GLOSSARY[str(n)] = f"{n} generic mana"
for a, b in HYBRID_PAIRS:
    SYMBOL_GLOSSARY[f"{a}/{b}"] = f"{COLOR_NAMES[a]} or {COLOR_NAMES[b]} mana"
for c, name in COLOR_NAMES.items():
    SYMBOL_GLOSSARY[f"2/{c}"] = f"two generic mana or one {name} mana"
    SYMBOL_GLOSSARY[f"C/{c}"] = f"one colorless mana or one {name} mana"
    SYMBOL_GLOSSARY[f"{c}/P"] = f"one {name} mana or 2 life"
for a, b in HYBRID_PAIRS:
    SYMBOL_GLOSSARY[f"{a}/{b}/P"] = f"{COLOR_NAMES[a]} mana, {COLOR_NAMES[b]} mana, or 2 life"


def expand_symbols(text: str) -> str:
    if not text:
        return text
    text = re.sub(r"\{([^{}]+)\}", lambda m: " " + SYMBOL_GLOSSARY.get(m.group(1), m.group(0)) + " ", text)
    return re.sub(r"\s{2,}", " ", text).strip()


print(f"{len(SYMBOL_GLOSSARY)} simbolos mapeados a partir da regra 107")
print(expand_symbols("{T}: Add {G}. Trample {2}{R}."))

66 simbolos mapeados a partir da regra 107
tap this permanent : Add green mana . Trample 2 generic mana red mana .


## Carregar o dicionario e a lista de ability words

In [4]:
with open(KEYWORD_DICT_PATH, encoding="utf-8") as f:
    kw_dict = json.load(f)
with open(ABILITY_WORDS_PATH, encoding="utf-8") as f:
    ability_words = set(json.load(f))

print(f"{len(kw_dict)} termos no dicionario, {len(ability_words)} ability words")

760 termos no dicionario, 61 ability words


## Casar as keywords da carta com o dicionario

Testando contra os dados reais, o campo `keywords` do Scryfall nem sempre
bate letra por letra com as chaves do dicionario:
- **Diferenca de maiuscula/minuscula** (`"First strike"` vs `"First
  Strike"`): tratada de forma automatica via `.lower()` -- isso e' uma
  normalizacao generica correta, nao uma tabela de excecao disfarcada, e
  nao faria sentido documentar cada um dos ~200 pares manualmente.
- **Tipos de token** (`"Treasure"` vs `"Treasure Token"`): esses SIM sao
  excecoes irregulares de verdade (nomes diferentes, nao so capitalizacao),
  entao viram uma tabela explicita `TOKEN_ALIASES` -- auditavel e facil de
  estender se aparecer um token novo.

In [5]:
TOKEN_ALIASES = {
    "treasure": "Treasure Token",
    "food": "Food Token",
    "clue": "Clue Token",
    "blood": "Blood Token",
    "map": "Map Token",
    "gold": "Gold Token",
    "junk": "Junk Token",
    "powerstone": "Powerstone Token",
    "incubator": "Incubator Token",
    "role token": "Role",
}


def build_keyword_lookup(kw_dict: dict):
    lower_map = {k.lower(): k for k in kw_dict}

    def lookup(keyword):
        if keyword in kw_dict:
            return kw_dict[keyword]
        if keyword.lower() in lower_map:
            return kw_dict[lower_map[keyword.lower()]]
        if keyword.lower() in TOKEN_ALIASES:
            alias = TOKEN_ALIASES[keyword.lower()]
            if alias in kw_dict:
                return kw_dict[alias]
        # fallback: variantes parametrizadas tipo "Hexproof from" que o
        # Scryfall registra como keyword propria (nao "Hexproof" + um
        # parametro separado) -- tenta a base antes de "from"/"with".
        for sep in (" from", " with"):
            if sep in keyword.lower():
                base = keyword[: keyword.lower().index(sep)].strip()
                if base in kw_dict:
                    return kw_dict[base]
                if base.lower() in lower_map:
                    return kw_dict[lower_map[base.lower()]]
        return None

    return lookup


lookup = build_keyword_lookup(kw_dict)

## Extrair o parametro especifico da keyword na carta

Uma definicao generica de `Cycling` nao diz que ESSA carta especifica tem
`Cycling {2}`, e a definicao generica de `Protection` nao diz que essa
carta da protecao especificamente contra preto. `extract_keyword_instance`
procura o trecho da carta logo apos o nome da keyword, ate a proxima
pontuacao/parenteses de reminder text, pra capturar esse parametro quando
existe.

**Limitacao conhecida, testada e documentada em vez de escondida:**
funciona bem pra keywords com parametro escrito logo apos o nome (Cycling
{2}, Ward {4}, Protection from black, Kicker {1}{G}). NAO funciona pra
`Landwalk`, porque o Scryfall guarda o nome generico no campo `keywords`
mas o `oracle_text` da carta usa o nome especifico (`Swampwalk`,
`Islandwalk`...) -- nesse caso a funcao simplesmente nao acha nada e a
`instance` fica `None`, sem quebrar o resto do pipeline.

In [6]:
def extract_keyword_instance(keyword: str, card_text: str):
    pattern = re.compile(rf"\b{re.escape(keyword)}\b[^\n,.;(]*", re.IGNORECASE)
    m = pattern.search(card_text)
    if not m:
        return None
    span = m.group(0).strip()
    if span.lower() == keyword.lower():
        return None  # sem parametro alem do proprio nome
    return span

## Montar as representacoes de texto pra embedding

A auditoria apontou dois riscos opostos: (1) so `oracle_text` nao da ao
embedding acesso a tipo/cor/custo; (2) injetar a definicao inteira de 10
keywords numa carta pode fazer o embedding representar mais o vocabulario
generico das regras do que a identidade especifica da carta.

Geramos duas variantes (nao mais tres -- `expanded` foi removida por ser
redundante com `structured`, so preenchia espaco sem trazer uma
comparacao realmente diferente):
- **baseline**: nome + tipo + custo + cores + oracle text (sem keywords) --
  controle barato pra comparacao futura
- **structured**: mesmos dados, em blocos rotulados (`Name:`, `Type:`,
  ...) + definicoes de keyword -- a usada pra gerar os embeddings

In [7]:
def build_keyword_definitions(card: dict, card_text: str):
    """Usada so internamente pra montar o texto estruturado -- nao e' mais
    salva no registro final (o conteudo dela ja fica dentro do texto)."""
    definitions = []
    for kw in card.get("keywords") or []:
        entry = lookup(kw)
        if entry:
            definitions.append({
                "keyword": kw,
                "instance": extract_keyword_instance(kw, card_text),
                "definition": entry["definition"],
                "source": entry["source"],
                "category": entry["category"],
            })
    return definitions


def build_embedding_texts(card: dict, card_text: str, keyword_defs: list):
    name = card.get("name") or ""
    type_line = card.get("type_line") or ""
    mana_cost = expand_symbols(card.get("mana_cost") or "")
    colors = ", ".join(COLOR_NAMES.get(c, c) for c in (card.get("colors") or []))
    oracle = expand_symbols(card_text)

    # O Modelo prioriza o que vem antes, então a ordem de acrescentar ao sections importa!
    baseline = f"{type_line}. {mana_cost}. {colors}. {oracle} {name}.".strip()

    sections = []
    if oracle:
        sections.append(f"Oracle Text:\n{oracle}")
    if keyword_defs:
        kw_lines = "\n".join(
            f"- {d['keyword']}" + (f" ({d['instance']})" if d["instance"] else "") + f": {d['definition']}"
            for d in keyword_defs
        )
        sections.append(f"Keyword Definitions:\n{kw_lines}")
    if mana_cost:
        sections.append(f"Mana Cost: {mana_cost}")
    if colors:
        sections.append(f"Colors: {colors}")
    sections.append(f"Type: {type_line}")
    sections.append(f"Name: {name}")
    structured = "\n".join(sections)

    return baseline, structured

## Processar todas as cartas (um unico passe pelo arquivo)

Cartas com `layout` em `EXCLUDED_LAYOUTS` ou `set_type` em
`EXCLUDED_SET_TYPES` sao puladas antes de qualquer processamento -- nao
entram no corpus de busca (Un-sets removidos por decisao revisada).

O registro final nao guarda `id`/`oracle_id` nem `keyword_definitions` --
o nome vira a referencia da carta, e as definicoes ja estao dentro do
texto estruturado. Os misses de keyword continuam classificados em A
(ability word)/B (gap real)/C (token especial), igual antes.

In [8]:
def classify_miss(keyword: str) -> str:
    # normaliza apostrofo reto vs curvo -- Scryfall usa reto ('), a CR usa
    # curvo (’) no texto oficial, e sao caracteres diferentes na comparacao
    kw_lower = keyword.lower().replace("’", "'")
    ability_words_norm = {w.replace("’", "'") for w in ability_words}
    if kw_lower in ability_words_norm or any(aw.startswith(kw_lower + " ") for aw in ability_words_norm):
        return "A_ability_word"
    if "token" in kw_lower:
        return "C_token_especial"
    return "B_gap_real"


missing_counter = {}
missing_by_category = {"A_ability_word": 0, "B_gap_real": 0, "C_token_especial": 0}
total_kw_mentions = 0
n_cards = 0
n_excluded = 0

with open(CARDS_PATH, encoding="utf-8") as fin, open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        card = json.loads(line)
        if card.get("layout") in EXCLUDED_LAYOUTS or card.get("set_type") in EXCLUDED_SET_TYPES:
            n_excluded += 1
            continue

        card_text = get_card_text(card)
        keywords = card.get("keywords") or []
        total_kw_mentions += len(keywords)

        keyword_defs = build_keyword_definitions(card, card_text)
        matched_names = {d["keyword"] for d in keyword_defs}
        for kw in keywords:
            if kw not in matched_names:
                missing_counter[kw] = missing_counter.get(kw, 0) + 1
                missing_by_category[classify_miss(kw)] += 1

        baseline, structured = build_embedding_texts(card, card_text, keyword_defs)

        record = {
            "name": card.get("name"),
            "type_line": card.get("type_line"),
            "mana_cost": card.get("mana_cost"),
            "colors": card.get("colors"),
            "power": card.get("power"),
            "toughness": card.get("toughness"),
            "loyalty": card.get("loyalty"),
            "defense": card.get("defense"),
            "produced_mana": card.get("produced_mana"),
            "oracle_text": card_text,
            "faces": get_card_faces(card),
            "keywords": keywords,
            "embedding_text_baseline": baseline,
            "embedding_text_structured": structured,
        }
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        n_cards += 1

n_missing = sum(missing_counter.values())
print(f"{n_excluded} cartas excluidas (layout/set_type fora do escopo de baralho)")
print(f"{n_cards} cartas processadas, salvas em {OUTPUT_PATH}")
print(f"{total_kw_mentions - n_missing}/{total_kw_mentions} mencoes de keyword expandidas ({(total_kw_mentions - n_missing)/total_kw_mentions*100:.1f}%)")
print()
print("classificacao dos misses:")
for cat, n in missing_by_category.items():
    print(f"  {cat}: {n} mencoes")

5479 cartas excluidas (layout/set_type fora do escopo de baralho)
33147 cartas processadas, salvas em Dados/Etapa1_card_expanded/card_documents.jsonl
21413/23623 mencoes de keyword expandidas (90.6%)

classificacao dos misses:
  A_ability_word: 1248 mencoes
  B_gap_real: 962 mencoes
  C_token_especial: 0 mencoes


## Conferindo alguns exemplos

In [9]:
with open(OUTPUT_PATH, encoding="utf-8") as f:
    cards = [json.loads(l) for l in f]
by_name = {c["name"]: c for c in cards}

print("=== confirmando que a Llanowar Elves token sumiu (era o problema original) ===")
llanowar_matches = [c for c in cards if c["name"] == "Llanowar Elves"]
print(f"{len(llanowar_matches)} registro(s) chamado(s) Llanowar Elves:", [c["type_line"] for c in llanowar_matches])
print()

print("=== confirmando os duplicados conhecidos do Unstable (esperado, nao e bug) ===")
everythingamajig = [c for c in cards if c["name"] == "Everythingamajig"]
print(f"{len(everythingamajig)} registros chamados Everythingamajig (fica so o ultimo em by_name!)")
print()

print("=== carta de dupla face, faces preservadas ===")
dfc = by_name["Ulvenwald Captive // Ulvenwald Abomination"]
for face in dfc["faces"]:
    print(" -", face["name"], "|", face["oracle_text"][:60])
print()

print("=== as duas representacoes, mesma carta ===")
sample = next(c for c in cards if c["name"] == "Llanowar Elves" and "Token" not in c["type_line"])
print("--- baseline ---")
print(sample["embedding_text_baseline"])
print("--- structured ---")
print(sample["embedding_text_structured"])
print()
print("campos no registro:", list(sample.keys()))

=== confirmando que a Llanowar Elves token sumiu (era o problema original) ===
1 registro(s) chamado(s) Llanowar Elves: ['Creature — Elf Druid']

=== confirmando os duplicados conhecidos do Unstable (esperado, nao e bug) ===
0 registros chamados Everythingamajig (fica so o ultimo em by_name!)

=== carta de dupla face, faces preservadas ===
 - Ulvenwald Captive | Defender
{T}: Add {G}.
{5}{G}{G}: Transform this creature.
 - Ulvenwald Abomination | {T}: Add {C}{C}.

=== as duas representacoes, mesma carta ===
--- baseline ---
Creature — Elf Druid. green mana. green. tap this permanent : Add green mana . Llanowar Elves.
--- structured ---
Oracle Text:
tap this permanent : Add green mana .
Mana Cost: green mana
Colors: green
Type: Creature — Elf Druid
Name: Llanowar Elves

campos no registro: ['name', 'type_line', 'mana_cost', 'colors', 'power', 'toughness', 'loyalty', 'defense', 'produced_mana', 'oracle_text', 'faces', 'keywords', 'embedding_text_baseline', 'embedding_text_structured']
